# Notebook 2: Bayes-Theorem

## Lernziele
- Verständnis des Bayes-Theorems und bedingter Wahrscheinlichkeiten
- Anwendung auf reale Beispiele wie medizinische Tests und Spamfilter
- Einführung in das Naive Bayes Modell für Klassifikation
- Vergleich Bayesianische vs. Frequentistische Statistik

## Inhalte

### 1. Bayes-Theorem: Formel & Intuition
- Formel: \( P(A|B) = \frac{P(B|A) P(A)}{P(B)} \)  
- Einfache Beispiele zur Visualisierung bedingter Wahrscheinlichkeiten
- Interaktive Darstellung von Prior, Likelihood und Posterior

### 2. Praxisbeispiele
- Medizinische Tests: Sensitivität, Spezifität, Prävalenz → Posterior-Wahrscheinlichkeit  
- Spamfilter: Wörter als Merkmale, Wahrscheinlichkeit für Spam vs. Nicht-Spam  

### 3. Naive Bayes Modell
- Bedingte Unabhängigkeit der Merkmale  
- Formeln zur Klassifikationsentscheidung  
- Python-Implementierung mit `sklearn.naive_bayes`  

### 4. Klassifikationsaufgabe
- Textklassifikation: Hund vs. Katze
- Schritte: Vektorisierung, One-Hot-Encoding → Naive Bayes → Evaluation  

### 5. Exkurs: Bayesianische vs. Frequentistische Statistik
- Frequentist: Wahrscheinlichkeiten = Grenzwert von Häufigkeiten, keine Priors  
- Bayesian: Wahrscheinlichkeiten = Grad des Glaubens / Wissen, Priors werden aktualisiert  
- Vergleich: Posterior vs. Maximum-Likelihood-Schätzer  
- Intuition: Bayes erlaubt Vorwissen und laufende Aktualisierung durch Daten


## 1. Bayes-Theorem: Formel & Intuition

Der **Satz von Bayes** erlaubt es, die Wahrscheinlichkeit eines Ereignisses $A$ zu aktualisieren, sobald wir neue Informationen $B$ erhalten. 

### Bedingte Wahrscheinlichkeit
Die bedingte Wahrscheinlichkeit $P(A|B)$ beschreibt, wie wahrscheinlich Ereignis $A$ eintritt, **unter der Bedingung**, dass $B$ bereits eingetreten ist:

$$
P(A|B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0
$$

### Bayes-Theorem
Bayes verbindet Vorwissen (Prior) mit neuer Evidenz (B), um die **aktualisierte Wahrscheinlichkeit** (Posterior) zu berechnen:

$$
P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}
$$

- $P(A)$: **Prior** – Wahrscheinlichkeit von $A$ vor Beobachtung von $B$  
- $P(B|A)$: **Likelihood** – Wahrscheinlichkeit, dass $B$ eintritt, wenn $A$ wahr ist  
- $P(B)$: **Normierungsfaktor** – Gesamte Wahrscheinlichkeit, dass $B$ eintritt  
- $P(A|B)$: **Posterior** – aktualisierte Wahrscheinlichkeit von $A$ nach Beobachtung von $B$  

> Hinweis: $A^c$ oder $B^c$ bezeichnet das **Komplement** eines Ereignisses, d.h. dass das Ereignis **nicht** eintritt.

### Beispiel: Corona-Test

- $A$: Person ist infiziert  
- $B$: Test ist positiv  
- Ziel: $P(A|B)$ – Wahrscheinlichkeit einer Infektion, wenn der Test positiv ist

Annahmen:  
- Prävalenz: $P(A) = 0.01$ (1 % der Bevölkerung infiziert)  
- Sensitivität: $P(B|A) = 0.95$ (95 % der Infizierten testen positiv)  
- Spezifität: $P(B^c|A^c) = 0.98$ (98 % der Nicht-Infizierten testen negativ)  
  - Somit $P(B|A^c) = 0.02$ (2% falsche positive Rate)

Berechnung der Gesamtwahrscheinlichkeit für einen positiven Test:

$$
P(B) = P(B|A)P(A) + P(B|A^c)P(A^c) = 0.95 \cdot 0.01 + 0.02 \cdot 0.99 = 0.0293
$$

Posterior (aktualisierte Wahrscheinlichkeit):

$$
P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)} = \frac{0.95 \cdot 0.01}{0.0293} \approx 0.324
$$

**Interpretation:** Trotz eines positiven Tests liegt die tatsächliche Wahrscheinlichkeit einer Infektion nur bei etwa 32 %, da die Prävalenz niedrig ist und es eine kleine Fehlerrate gibt.  

**Kernaussage:** Bayes ermöglicht es, Vorwissen (Prävalenz) und neue Evidenz (Testergebnis) korrekt zu kombinieren, um die realistische Wahrscheinlichkeit abzuschätzen.


In [105]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def bayes_visual_de(prior=0.01, sensitivity=0.95, specificity=0.98):
    # Komplement
    p_nicht = 1 - prior
    
    # Falsche Positive
    falsch_pos = 1 - specificity
    
    # Gesamtwahrscheinlichkeit eines positiven Tests
    p_test_pos = sensitivity * prior + falsch_pos * p_nicht
    
    # Posterior
    posterior = (sensitivity * prior) / p_test_pos
    
    labels = ['Ereignis tritt ein', 'Ereignis tritt nicht ein']
    prior_vals = [prior, p_nicht]
    likelihood_vals = [sensitivity, falsch_pos]
    posterior_vals = [posterior, 1 - posterior]
    
    fig, ax = plt.subplots(1,3, figsize=(12,4))
    
    # Prior
    ax[0].bar(labels, prior_vals, color=['orange','lightblue'])
    ax[0].set_title('Vorkommen (Prior)')
    ax[0].set_ylim(0,1)
    for i, v in enumerate(prior_vals):
        ax[0].text(i, v+0.02, f"{v:.2f}", ha='center')
    
    # Likelihood (positiver Test)
    ax[1].bar(labels, likelihood_vals, color=['orange','lightblue'])
    ax[1].set_title('Treffer- bzw. Falsch-Positiv-Wahrscheinlichkeit')
    ax[1].set_ylim(0,1)
    for i, v in enumerate(likelihood_vals):
        ax[1].text(i, v+0.02, f"{v:.2f}", ha='center')
    
    # Posterior
    ax[2].bar(labels, posterior_vals, color=['orange','lightblue'])
    ax[2].set_title('Posterior nach positivem Test')
    ax[2].set_ylim(0,1)
    for i, v in enumerate(posterior_vals):
        ax[2].text(i, v+0.02, f"{v:.2f}", ha='center')
    
    plt.show()

interact(
    bayes_visual_de,
    prior=FloatSlider(min=0.001, max=0.1, step=0.001, value=0.01, description='Vorkommen'),
    sensitivity=FloatSlider(min=0.5, max=1.0, step=0.01, value=0.95, description='Trefferwahrscheinlichkeit'),
    specificity=FloatSlider(min=0.5, max=1.0, step=0.01, value=0.98, description='Falsch-Positiv-Wahrscheinlichkeit')
)


interactive(children=(FloatSlider(value=0.01, description='Vorkommen', max=0.1, min=0.001, step=0.001), FloatS…

<function __main__.bayes_visual_de(prior=0.01, sensitivity=0.95, specificity=0.98)>

## 3. Naive Bayes Klassifikator (vereinfachte Version)

Um den Naive Bayes Klassifikator einfach einzuführen, betrachten wir zunächst **zwei Klassen** \(A\) und \(B\) (z. B. `Spam` und `Nicht-Spam`) und ein Merkmal \(X\), das nur zwei Zustände hat (z. B. das Wort „gratis“ ist in der E-Mail enthalten oder nicht).

### 3.1 Klassifikationsidee

Wir möchten vorhersagen, ob eine E-Mail zur Klasse \(A\) oder \(B\) gehört, basierend auf dem Merkmal \(X\):

$$
P(A \mid X) = \frac{P(A) \cdot P(X \mid A)}{P(X)}
$$

$$
P(B \mid X) = \frac{P(B) \cdot P(X \mid B)}{P(X)}
$$

**Klassifikationsregel:**

$$
\text{Vorhersage} = 
\begin{cases} 
A & \text{wenn } P(A \mid X) > P(B \mid X) \\
B & \text{sonst}
\end{cases}
$$

### 3.2 Intuition

- **Prior \(P(A)\), \(P(B)\)**: Die Wahrscheinlichkeit, dass ein Objekt zu A bzw. B gehört, **ohne Wissen über das Merkmal**.  

- **Likelihood \(P(X \mid A)\), \(P(X \mid B)\)**: Die Wahrscheinlichkeit, dass das Merkmal \(X\) auftritt, **wenn die Klasse bekannt ist**.  

- **Posterior \(P(A \mid X)\), \(P(B \mid X)\)**: Die **aktualisierte Wahrscheinlichkeit**, dass das Objekt zu A bzw. B gehört, **nachdem das Merkmal \(X\) beobachtet wurde**, berechnet mit dem Satz von Bayes:  

$$
P(A \mid X) = \frac{P(X \mid A) P(A)}{P(X)} , \quad
P(B \mid X) = \frac{P(X \mid B) P(B)}{P(X)}
$$

- **Normierungsfaktor \(P(X)\)**: Gesamte Wahrscheinlichkeit, dass das Merkmal \(X\) auftritt:

$$
P(X) = P(X \mid A) P(A) + P(X \mid B) P(B)
$$


### 3.3 Beispiel

- 100 E-Mails: 40 Spam (A), 60 Nicht-Spam (B)  
- Merkmal: Wort „gratis“  
  - 25 Spam-Mails enthalten „gratis“ → \(P(X=1 | A) = 25/40 = 0.625\)  
  - 5 Nicht-Spam-Mails enthalten „gratis“ → \(P(X=1 | B) = 5/60 $\approx$ 0.083\)

Neue E-Mail enthält „gratis“ → Berechne Posterior:

$$
P(A \mid X=1) \propto P(A) \cdot P(X=1 \mid A) = 0.4 \cdot 0.625 = 0.25
$$

$$
P(B \mid X=1) \propto P(B) \cdot P(X=1 \mid B) = 0.6 \cdot 0.083 \approx 0.05
$$

**Vorhersage:** Klasse A (`Spam`), da \(P(A \mid X) > P(B \mid X)\).

### 3.4 Python-Beispiel

```python
# Binäre Merkmale und Klassen
P_A = 0.4
P_B = 0.6
P_X_given_A = 0.625
P_X_given_B = 0.083

# Posterior berechnen
posterior_A = P_A * P_X_given_A
posterior_B = P_B * P_X_given_B

klassifizierung = "A (Spam)" if posterior_A > posterior_B else "B (Nicht-Spam)"
print("Vorhergesagte Klasse:", klassifizierung)


In [106]:
import ipywidgets as widgets
from IPython.display import display

def naive_bayes_interaktiv(prior_spam=0.4, likelihood_word_spam=0.6, likelihood_word_nonspam=0.1):
    prior_nonspam = 1 - prior_spam
    
    posterior_spam = prior_spam * likelihood_word_spam
    posterior_nonspam = prior_nonspam * likelihood_word_nonspam
    
    # Normierung (optional)
    posterior_spam_norm = posterior_spam / (posterior_spam + posterior_nonspam)
    posterior_nonspam_norm = posterior_nonspam / (posterior_spam + posterior_nonspam)
    
    print(f"Posterior Spam: {posterior_spam_norm:.2f}")
    print(f"Posterior Nicht-Spam: {posterior_nonspam_norm:.2f}")
    klasse = "Spam" if posterior_spam_norm > posterior_nonspam_norm else "Nicht-Spam"
    print(f"Vorhersage: {klasse}")

# Interaktive Slider
widgets.interact(
    naive_bayes_interaktiv,
    prior_spam=widgets.FloatSlider(min=0, max=1, step=0.05, value=0.4, description='Prior Spam'),
    likelihood_word_spam=widgets.FloatSlider(min=0, max=1, step=0.05, value=0.6, description='Wahrscheinlichkeit Wort|Spam'),
    likelihood_word_nonspam=widgets.FloatSlider(min=0, max=1, step=0.05, value=0.1, description='Wahrscheinlichkeit Wort|Nicht-Spam')
)


interactive(children=(FloatSlider(value=0.4, description='Prior Spam', max=1.0, step=0.05), FloatSlider(value=…

<function __main__.naive_bayes_interaktiv(prior_spam=0.4, likelihood_word_spam=0.6, likelihood_word_nonspam=0.1)>

## 3.4 Vollständiger Naive Bayes Klassifikator – Binäres Klassifikationsbeispiel

Hier zeigen wir, wie der **Naive Bayes Klassifikator** auf ein **binäres Klassifikationsproblem** (Spam vs. Nicht-Spam) angewendet wird. Wir verwenden dabei eine einfache Textklassifikation mit wenigen Merkmalen.

> **Hinweis:** Der Klassifikator geht davon aus, dass das Vorkommen jedes Wortes **unabhängig von anderen Wörtern** ist (Naive Annahme).

### 3.4.1 Merkmale und Klassen

- **Merkmale** $X_1, X_2, ..., X_n$: Indikatoren, ob bestimmte Wörter in einer E-Mail vorkommen (z.B. "Sonderangebot", "jetzt", "Treffen").
- **Klassen**:  
  - $A = \text{Spam}$  
  - $B = \text{Nicht-Spam}$

Ziel: Berechne $P(A \mid X)$ und $P(B \mid X)$ für eine neue E-Mail.

### 3.4.2 Klassifikationsregel

1. **Berechne Priors**: $P(A)$, $P(B)$ aus Trainingsdaten  
2. **Berechne Likelihoods**: $P(X_i \mid A)$, $P(X_i \mid B)$ für jedes Merkmal  
3. **Posterior berechnen**:

$$
P(A \mid X) = \frac{P(A) \prod_i P(X_i \mid A)}{P(X)}, \quad
P(B \mid X) = \frac{P(B) \prod_i P(X_i \mid B)}{P(X)}
$$

4. **Klassenzuordnung**:

$$
\hat{C} = 
\begin{cases}
A, & \text{wenn } P(A \mid X) > P(B \mid X) \\
B, & \text{sonst}
\end{cases}
$$

### 3.4.3 Python-Implementierung (binär, kleiner Datensatz)

```python
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

# Trainingsdaten
emails = [
    "Kaufen Sie jetzt!",            # Spam
    "Sonderangebot nur heute!",     # Spam
    "Treffen wir uns morgen?",      # Nicht-Spam
    "Hallo, wie geht's?"            # Nicht-Spam
]
labels = [1, 1, 0, 0]  # 1=Spam, 0=Nicht-Spam

# Merkmalsmatrix
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(emails)

# Naive Bayes Klassifikator
clf = MultinomialNB()
clf.fit(X, labels)

# Neue E-Mail klassifizieren
neue_email = ["Jetzt Sonderangebot"]
X_new = vectorizer.transform(neue_email)
posterior_prob = clf.predict_proba(X_new)[0]

print("Posterior Wahrscheinlichkeit Spam:", posterior_prob[1])
print("Posterior Wahrscheinlichkeit Nicht-Spam:", posterior_prob[0])
print("Vorhergesagte Klasse:", "Spam" if posterior_prob[1] > posterior_prob[0] else "Nicht-Spam")


## 4. Klassifikationsaufgabe: Tierklassifikation

In dieser Übung verwenden wir den **Naive Bayes Klassifikator**, um Tiere als **Katze** oder **Hund** zu klassifizieren. Anders als beim klassischen Spam-Beispiel werden hier die Merkmale **korreliert**, was zu einer fehlerhaften Klassifikation führen kann.

### 4.1 Aufgabenstellung

Gegeben ist ein kleiner Datensatz von Tieren mit den folgenden Merkmalen:

- Fellfarbe: `schwarz`, `braun`, `weiß`
- Ohrform: `spitz`, `rund`
- Schwanzlänge: `kurz`, `lang`

**Ziel:** Vorhersage, ob es sich um eine **Katze** oder einen **Hund** handelt.

### 4.2 Besonderheit

**Aufgabe:** Klassifiziere die Tiere (Katze vs. Hund) anhand der Merkmale:

- Fellfarbe: `schwarz`, `braun`, `weiß`
- Ohrform: `spitz`, `rund`
- Schwanzlänge: `kurz`, `lang`

> Hinweis: Bei Katzen treten bestimmte Kombinationen häufiger auf, z. B. `schwarz` oder `weiß` **mit spitzen Ohren**. Bei Hunden andere Kombinationen.  
> Beobachtet, wie der Naive Bayes Klassifikator reagiert, ohne dass wir die Korrelation explizit erklären.

**Ziel:**  
- Erst ausprobieren, wie der Klassifikator arbeitet.  
- Danach analysieren, warum die Vorhersagen fehlerhaft sind, wenn Merkmale stark korreliert sind.  
- Awareness schaffen: Vor dem Einsatz eines Modells die **Feature-Abhängigkeiten prüfen**.


### 4.3 Schritte

1. **Datensatz erstellen**  
   - Kleine Tabelle mit Tierbeispielen und Merkmalen.
2. **Merkmalsvektor erstellen**  
   - Kodierung der Merkmale in numerische Form (z. B. `One-Hot-Encoding`).
3. **Naive Bayes Klassifikator trainieren**  
   - `sklearn.naive_bayes` verwenden.
4. **Vorhersage treffen**  
   - Testdaten klassifizieren.
5. **Evaluation und Visualisierung**  
   - Vergleich von Vorhersage und tatsächlicher Klasse.
   - Analyse: Welche Fehler entstehen durch korrelierte Merkmale?


### 4.3.1 Datensatz erstellen

In [107]:
import pandas as pd
import numpy as np

# Zufallsgenerator für Reproduzierbarkeit
np.random.seed(42)

# Anzahl Tiere
n_cats = 50
n_dogs = 50

# Katzen: Fellfarbe und Ohrform korreliert (schwarz oder weiß → spitze Ohren)
cat_colors = np.random.choice(['schwarz', 'weiß', 'braun'], size=n_cats, p=[0.4, 0.4, 0.2])
cat_ears = [ 'spitz' if c in ['schwarz','weiß'] else 'rund' for c in cat_colors ]
cat_tails = np.random.choice(['kurz','lang'], size=n_cats)

# Hunde: Fellfarbe und Ohrform korreliert anders (braun oder schwarz → runde Ohren)
dog_colors = np.random.choice(['schwarz','braun','weiß'], size=n_dogs, p=[0.3,0.5,0.2])
dog_ears = [ 'rund' if c in ['braun','schwarz'] else 'spitz' for c in dog_colors ]
dog_tails = np.random.choice(['kurz','lang'], size=n_dogs)

# DataFrame zusammenstellen
df = pd.DataFrame({
    'Tier': ['Katze']*n_cats + ['Hund']*n_dogs,
    'Fellfarbe': np.concatenate([cat_colors, dog_colors]),
    'Ohrform': np.concatenate([cat_ears, dog_ears]),
    'Schwanz': np.concatenate([cat_tails, dog_tails])
})

# Shuffle the dataset for randomness
df = df.sample(frac=1).reset_index(drop=True)

# Ausgabe des ersten Teils des Datensatzes
df.head(10)


,Tier,Fellfarbe,Ohrform,Schwanz
0,Katze,schwarz,spitz,lang
1,Hund,braun,rund,kurz
2,Hund,braun,rund,lang
3,Hund,braun,rund,lang
4,Katze,weiß,spitz,kurz
5,Hund,braun,rund,kurz
6,Hund,weiß,spitz,kurz
7,Katze,weiß,spitz,kurz
8,Hund,braun,rund,lang
9,Katze,schwarz,spitz,lang


### 4.3.2 One-Hot-Encoding

**Zweck:**  
Kategoriale Merkmale (z. B. Farben, Typen, Kategorien) in eine **numerische Form** bringen, damit Algorithmen wie Naive Bayes, lineare Modelle oder neuronale Netze sie verarbeiten können.

**Funktionsweise:**  
- Für jede mögliche Kategorie eines Merkmals wird eine **neue Spalte** erstellt.  
- Der Wert `1` zeigt an, dass das Objekt diese Kategorie hat, `0` bedeutet, dass es diese Kategorie nicht hat.  

**Beispiel:**  

| Fellfarbe  | schwarz | weiß | braun |
|------------|---------|------|-------|
| schwarz    | 1       | 0    | 0     |
| weiß       | 0       | 1    | 0     |
| braun      | 0       | 0    | 1     |

**Vorteile:**  
- Macht kategoriale Daten **modellierbar** für ML-Algorithmen.  
- Bewahrt die **Information jeder Kategorie**, ohne falsche Ordinalbeziehungen zu implizieren.

**Hinweis:**  
- Korrelationen zwischen Merkmalen bleiben in den One-Hot-Spalten bestehen.  
- Für sehr viele Kategorien kann die Anzahl der Spalten stark wachsen (**High-Dimensionalität**).  


In [108]:
# Ausgangsdaten
data = df.copy()
data.rename(columns={'Tier': 'Tierart'}, inplace=True)

# Manuelles One-Hot-Encoding der vorhandenen Merkmale
Fellfarbe_encoded = pd.get_dummies(data['Fellfarbe'], prefix='Fellfarbe')
Ohrform_encoded = pd.get_dummies(data['Ohrform'], prefix='Ohrform')
Schwanz_encoded = pd.get_dummies(data['Schwanz'], prefix='Schwanz')

# Alle Features zusammenführen
X_encoded = pd.concat([Fellfarbe_encoded, Ohrform_encoded, Schwanz_encoded], axis=1)
y = data['Tierart']

print("One-Hot-Encoded Merkmalsmatrix:")
print(X_encoded.head())


One-Hot-Encoded Merkmalsmatrix:
   Fellfarbe_braun  Fellfarbe_schwarz  Fellfarbe_weiß  Ohrform_rund  \
0            False               True           False         False   
1             True              False           False          True   
2             True              False           False          True   
3             True              False           False          True   
4            False              False            True         False   

   Ohrform_spitz  Schwanz_kurz  Schwanz_lang  
0           True         False          True  
1          False          True         False  
2          False         False          True  
3          False         False          True  
4           True          True         False  


### 4.3.3 Trainieren des Naive Bayes Klassifizierers

### 4.3.3 Klassifizierung mit Naive Bayes Klassifizierer

In diesem Schritt wenden wir den **Naive Bayes Klassifikator** auf unseren künstlich erzeugten Datensatz an. Die Vorgehensweise ist wie folgt:

1. **Merkmalsvektor vorbereiten**  
   - Wir verwenden die One-Hot-Encoded Version der Merkmale `Fellfarbe`, `Ohrform` und `Schwanz`.  
   - Jedes Merkmal wird in binäre Indikatorvariablen umgewandelt.

2. **Zielvariable definieren**  
   - Die Zielvariable `y` ist die Tierart (`Katze` oder `Hund`).

3. **Modell trainieren**  
   - Wir verwenden `sklearn.naive_bayes.MultinomialNB` für die Klassifikation.  
   - Der Klassifikator lernt die **Wahrscheinlichkeiten jedes Merkmals gegeben der Klasse**, basierend auf den Trainingsdaten.

4. **Vorhersagen machen**  
   - Neue Datenpunkte werden klassifiziert, indem die **Posterior-Wahrscheinlichkeiten** berechnet werden:  
     $$
     P(\text{Tierart} \mid \text{Merkmale}) \propto P(\text{Tierart}) \prod_i P(\text{Merkmal}_i \mid \text{Tierart})
     $$

**Beispiel One-Hot-Encoded Vektor:**  

Ein schwarzer Hund mit runden Ohren und kurzem Schwanz könnte wie folgt codiert sein:

| Fellfarbe_schwarz | Fellfarbe_weiss | Fellfarbe_braun | Ohrform_spitz | Ohrform_rund | Schwanz_kurz | Schwanz_lang |
|------------------|----------------|----------------|---------------|--------------|--------------|--------------|
| 1                | 0              | 0              | 0             | 1            | 1            | 0            |



In [109]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Ausgangsdaten (dein erstellter DataFrame df)
data = df.copy()
data.rename(columns={'Tier': 'Tierart'}, inplace=True)

# Manuelles One-Hot-Encoding
Fellfarbe_encoded = pd.get_dummies(data['Fellfarbe'], prefix='Fellfarbe')
Ohrform_encoded = pd.get_dummies(data['Ohrform'], prefix='Ohrform')
Schwanz_encoded = pd.get_dummies(data['Schwanz'], prefix='Schwanz')

# Alle Features zusammenführen
X_encoded = pd.concat([Fellfarbe_encoded, Ohrform_encoded, Schwanz_encoded], axis=1)
y = data['Tierart']

# Trainings- und Testdaten aufteilen
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Naive Bayes Klassifikator
clf = MultinomialNB()
clf.fit(X_train, y_train)

# Vorhersage
y_pred = clf.predict(X_test)

# Evaluation
# print("Accuracy:", accuracy_score(y_test, y_pred))
#print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Beispiel: schwarzer Hund mit runden Ohren und kurzem Schwanz
example = pd.DataFrame({
    'Fellfarbe_braun': [0],
    'Fellfarbe_schwarz': [1],
    'Fellfarbe_weiß': [0],
    'Ohrform_rund': [1],
    'Ohrform_spitz': [0],
    'Schwanz_kurz': [1],
    'Schwanz_lang': [0]
})

pred_example = clf.predict(example)
print("Vorhersage:", pred_example[0])



Vorhersage: Hund


## 4.4 Demonstration der Fehleranfälligkeit

Der Naive Bayes Klassifikator nimmt an, dass alle Merkmale **unabhängig** voneinander sind, gegeben die Klasse.  
Wenn Merkmale **korreliert** sind, kann die Wahrscheinlichkeit überschätzt werden und die Klassifikation fehlschlagen.

### Ziel
- Zeige Beispiele von Tieren, deren Merkmale Merkmale beider Klassen widerspiegeln.
- Vergleiche Vorhersage des Klassifikators mit der tatsächlichen Tierart.


In [110]:
import ipywidgets as widgets
from IPython.display import display, HTML
import pandas as pd

# Feature-Optionen
fellfarbe_options = ['braun','schwarz','weiß']
ohrform_options = ['rund','spitz']
schwanz_options = ['kurz','lang']
klasse_options = ['Katze','Hund']

# Vordefinierte fehlerhafte Beispiele
examples_fail = pd.DataFrame([
    {'Fellfarbe':'braun','Ohrform':'spitz','Schwanz':'kurz','Tatsächlich':'Katze'},
    {'Fellfarbe':'schwarz','Ohrform':'spitz','Schwanz':'lang','Tatsächlich':'Hund'},
    {'Fellfarbe':'weiß','Ohrform':'rund','Schwanz':'kurz','Tatsächlich':'Katze'}
])

def show_example(selected_case, fellfarbe, ohrform, schwanz, tatsaechlich):
    if selected_case != 'Keiner (frei wählen)':
        # Vordefiniertes Beispiel übernehmen
        row = examples_fail.loc[int(selected_case.split()[-1])-1]
        fellfarbe, ohrform, schwanz, tatsaechlich = row['Fellfarbe'], row['Ohrform'], row['Schwanz'], row['Tatsächlich']
    
    # Manuelles One-Hot-Encoding des Beispiels
    example = pd.DataFrame([{
        'Fellfarbe_braun': int(fellfarbe=='braun'),
        'Fellfarbe_schwarz': int(fellfarbe=='schwarz'),
        'Fellfarbe_weiß': int(fellfarbe=='weiß'),
        'Ohrform_rund': int(ohrform=='rund'),
        'Ohrform_spitz': int(ohrform=='spitz'),
        'Schwanz_kurz': int(schwanz=='kurz'),
        'Schwanz_lang': int(schwanz=='lang')
    }])
    
    pred = clf.predict(example)[0]
    
    # Farbcode: grün = richtig, rot = falsch
    color = "green" if pred == tatsaechlich else "red"
    
    # Anzeige
    display(HTML(f"<h3 style='color:{color}'>Vorhersage: {pred} | Tatsächlich: {tatsaechlich}</h3>"))
    
    # Feature-Visualisierung
    feature_str = f"Fellfarbe: {fellfarbe} | Ohrform: {ohrform} | Schwanz: {schwanz}"
    display(HTML(f"<p><b>Features:</b> {feature_str}</p>"))

# Dropdown für vordefinierte Fälle + Option für freie Auswahl
case_dropdown = ['Keiner (frei wählen)', 'Fall 1', 'Fall 2', 'Fall 3']

widgets.interact(
    show_example,
    selected_case=widgets.Dropdown(options=case_dropdown, value='Keiner (frei wählen)', description='Beispiel:'),
    fellfarbe=widgets.Dropdown(options=fellfarbe_options, value='braun', description='Fellfarbe:'),
    ohrform=widgets.Dropdown(options=ohrform_options, value='rund', description='Ohrform:'),
    schwanz=widgets.Dropdown(options=schwanz_options, value='kurz', description='Schwanz:'),
    tatsaechlich=widgets.Dropdown(options=klasse_options, value='Katze', description='Tatsächliche Klasse:')
)


interactive(children=(Dropdown(description='Beispiel:', options=('Keiner (frei wählen)', 'Fall 1', 'Fall 2', '…

<function __main__.show_example(selected_case, fellfarbe, ohrform, schwanz, tatsaechlich)>